Es una API de **Universidad Digital** hecha con FastAPI, SQLAlchemy y PostgreSQL. No modifiqué nada. Abajo va qué hace cada archivo, siguiendo el orden en que el programa realmente se ejecuta.

## Qué resuelve

Gestiona usuarios con roles, materias, períodos académicos, inscripciones, calificaciones y un reporte de promedios. Cada dominio vive en su carpeta (`models` = tablas, `schemas` = validación de entrada y salida, `services` = reglas, `routes` = URLs). Las rutas no deciden el negocio: llaman al servicio y devuelven lo que este produce.

Hay tres roles fijos: `ADMINISTRADOR`, `DOCENTE` y `ESTUDIANTE`. Un usuario puede tener varios.

---

## Arranque

### `app/core/config.py`

Carga la configuración desde variables de entorno y, si existe, desde `.env`.

- `Settings` hereda de `BaseSettings`. `extra="ignore"` descarta variables que no estén declaradas.
- `db_host`, `db_port`, `db_name` y `db_user` tienen valores por defecto (`localhost`, `5432`, `universidad`).
- `db_password` es obligatoria (mínimo 1 carácter). `jwt_secret` es obligatoria y de al menos 32 caracteres.
- `access_token_minutes` vale 60 y solo acepta entre 5 y 1440.
- `cors_origins` por defecto es `http://localhost:5173`.
- `database_url` arma la URL `postgresql+psycopg` con usuario, clave, host, puerto y base.
- `allowed_origins` parte esa cadena por comas, quita espacios y descarta vacíos.
- `get_settings()` está decorada con `@lru_cache`: la primera llamada lee el entorno y las siguientes reutilizan el mismo objeto.

### `app/core/database.py`

- `Base` es la clase de la que heredan todos los modelos; SQLAlchemy registra ahí las tablas.
- `engine` abre el pool hacia PostgreSQL. `pool_pre_ping=True` prueba la conexión antes de usarla, para no reutilizar una que ya murió.
- `SessionLocal` fabrica sesiones. `expire_on_commit=False` deja los objetos usables después del `commit`.
- `get_db()` abre una sesión, la entrega a la ruta con `yield` y la cierra al terminar la petición.

### `app/core/models.py`

No define tablas. Importa `Role`, `User`, `Subject`, `AcademicPeriod`, `Enrollment` y `Grade` para que, cuando Alembic inspeccione `Base.metadata`, esas clases ya estén registradas.

### `app/core/security.py`

- `password_hash = PasswordHash.recommended()` usa el hasher recomendado de `pwdlib` (Argon2).
- `hash_password` convierte la contraseña en un hash. `verify_password` compara una contraseña en claro con ese hash.
- `create_access_token` arma un JWT HS256. El cuerpo lleva `sub` (el id del usuario como texto), `iat` (ahora en UTC) y `exp` (ahora más los minutos de configuración). Se firma con `jwt_secret`.

### `app/core/errors.py`

- `AppError` guarda un código HTTP, un código corto (`not_found`, `conflict`, `forbidden`) y un mensaje.
- `app_error_handler` convierte esa excepción en JSON: `{"error": {"code": "...", "message": "..."}}`. No incluye trazas ni datos internos.
- `not_found` devuelve 404, `conflict` 409 y `forbidden` 403 con el texto "No tiene permiso para esta operación".

### `app/main.py`

- Crea la app FastAPI con título y versión.
- Registra el manejador de `AppError`.
- Añade CORS: orígenes de la configuración, credenciales permitidas, métodos GET/POST/PUT/PATCH/DELETE y cabeceras `Authorization` y `Content-Type`.
- Monta los routers de auth, usuarios, materias, períodos, inscripciones, calificaciones y reportes bajo `/api/v1`.
- `GET /health/live` responde `{"status": "ok"}` sin tocar la base. Sirve para saber si el proceso está vivo.
- `GET /health/ready` ejecuta `SELECT 1`. Si la base responde, devuelve `{"database": "ready"}`.

### `app/run.py`

Si se ejecuta como programa (`python -m app.run`), arranca Uvicorn en `0.0.0.0` y en el puerto de la variable `PORT`, o 8000 si no existe. Es el comando del contenedor.

### `app/bootstrap.py`

Se corre una vez a mano (`python -m app.bootstrap`), no al arrancar la API.

- Define los tres roles y su descripción.
- Lee `ADMIN_EMAIL` (lo pasa a minúsculas y le quita espacios) y `ADMIN_PASSWORD`. Si falta el correo o la clave tiene menos de 12 caracteres, termina el proceso con ese mensaje.
- Abre una transacción. Por cada rol, si no existe en la tabla, lo inserta. `flush` escribe esos inserts antes de seguir, sin cerrar la transacción.
- Busca un usuario con ese correo. Si no existe, crea "Administración Inicial" con la contraseña hasheada y el rol `ADMINISTRADOR`.
- Si el usuario ya existía, no lo toca. Por eso se puede ejecutar más de una vez.
- Imprime que los roles y el administrador quedaron verificados.

---

## Autenticación

### `app/auth/schemas.py`

`TokenResponse` tiene `access_token` y `token_type`, que por defecto es `"bearer"`.

### `app/auth/routes.py`

`POST /api/v1/auth/token` recibe el formulario OAuth2 de usuario y contraseña (el campo se llama `username`, pero aquí se usa como correo). Pide una sesión y devuelve el token que produce `services.login`.

### `app/auth/services.py`

`login` busca el usuario por correo en minúsculas y sin espacios.

- Si no existe, está inactivo o la contraseña no coincide, lanza 401 con "Email o contraseña inválidos". El mismo mensaje cubre los tres casos, así no se revela cuál falló.
- Si pasa, devuelve el JWT con `user.id_usuario`.

### `app/auth/dependencies.py`

- `oauth2_scheme` exige el encabezado `Authorization: Bearer ...` y le dice a Swagger que el token se pide en `/api/v1/auth/token`.
- `current_user` decodifica el JWT con el secreto y el algoritmo HS256, toma `sub` y lo convierte a entero. Si el token es inválido, expiró, no trae `sub` o el id no es número, lanza 401. Luego carga el usuario. Si no existe o `activo` es falso, también 401. Si todo cuadra, devuelve el usuario.
- `require_roles("ADMINISTRADOR")` (u otros nombres) devuelve una función guardia. Esa función exige un usuario autenticado y comprueba si alguno de sus roles está en la lista. Si no, lanza 403. Si sí, deja pasar al usuario.

---

## Usuarios

### `app/users/models.py`

Tabla puente `usuario_rol`: pareja `id_usuario` + `id_rol`, ambas clave primaria. `ondelete="CASCADE"` borra la fila de la relación si se borra el usuario o el rol.

`Role` (`roles`): `id_rol`, `nombre` único de hasta 20 caracteres, `descripcion` de hasta 150.

`User` (`usuarios`): `id_usuario`, `nombres`, `apellidos`, `email` único, `password_hash`, `activo` (por defecto verdadero), `fecha_creacion` puesta por la base al insertar. `roles` es la relación muchos a muchos a través de `usuario_rol`. `lazy="selectin"` carga los roles en la misma consulta del usuario, para no hacer una consulta extra por cada uno.

### `app/users/schemas.py`

- `UserCreate`: nombres y apellidos de 1 a 100, correo válido, contraseña de 12 a 128, y al menos un rol. El validador deja el correo en minúsculas.
- `UserStatus`: solo un booleano `activo`.
- `UserRoles`: lista de roles con al menos uno.
- `UserResponse`: id, nombres, apellidos, correo, activo, fecha y roles. `from_attributes=True` permite construirlo desde el objeto de SQLAlchemy. No hay campo de contraseña ni de hash.

### `app/users/services.py`

`ROLE_NAMES` es el conjunto cerrado de los tres roles.

- `_roles` convierte los nombres pedidos en un conjunto. Si está vacío o trae un nombre fuera de esos tres, lanza 409. Busca esos roles en la base. Si falta alguno, lanza 409 pidiendo ejecutar el bootstrap. Devuelve los objetos `Role`.
- `to_response` copia los campos públicos y ordena los nombres de rol.
- `create_user` recorta espacios de nombres y apellidos, guarda el correo en minúsculas, hashea la contraseña y asigna los roles. Si el correo ya existe, la base lanza `IntegrityError`: se revierte la transacción y se responde 409.
- `get_user` busca por id. Si no está, 404.
- `list_users` ordena por id y aplica `offset` y `limit`.
- `set_status` cambia `activo` y guarda.
- `set_roles` reemplaza la lista completa de roles del usuario.

### `app/users/routes.py`

Prefijo `/users`. `Admin` es la dependencia que exige rol `ADMINISTRADOR`.

| Método y ruta | Quién | Qué hace |
|---|---|---|
| `GET /me` | cualquiera autenticado | su propio perfil |
| `POST /` | admin | crea usuario, responde 201 |
| `GET /` | admin | lista, `offset` desde 0 y `limit` de 1 a 100 (defecto 50) |
| `GET /{user_id}` | admin | un usuario |
| `PATCH /{user_id}/status` | admin | activa o desactiva |
| `PUT /{user_id}/roles` | admin | sustituye roles |

---

## Materias

### `app/subjects/models.py`

Tabla `materias`: id, nombre, código único, créditos, `activa` (verdadero por defecto) e `id_docente` opcional, que apunta a `usuarios`. `ondelete="RESTRICT"` impide borrar un usuario que siga asignado como docente. El check `creditos > 0` vive también en la base.

### `app/subjects/schemas.py`

- `SubjectCreate`: nombre 2–150, código 2–30 (se recorta y pasa a mayúsculas), créditos de 1 a 30, docente opcional y, si viene, mayor que 0.
- `SubjectUpdate` añade `activa`.
- `SubjectResponse` añade `id_materia` y `activa`.

### `app/subjects/services.py`

- `_validate_teacher`: si no hay docente, no hace nada. Si hay, el usuario debe existir, estar activo y tener rol `DOCENTE`. Si no, 409.
- `get_subject` busca por id o lanza 404.
- `list_subjects` pagina igual que usuarios.
- `create_subject` valida al docente, inserta y, si el código ya existe, 409.
- `update_subject` carga la materia, valida al docente y copia cada campo del cuerpo sobre el objeto.
- `deactivate_subject` pone `activa = False`. No borra la fila.

### `app/subjects/routes.py`

Listar y ver una materia exige estar autenticado, sin importar el rol. Crear, actualizar y desactivar exigen administrador. El `DELETE` no elimina: llama a `deactivate_subject` y devuelve la materia ya inactiva.

---

## Períodos

### `app/periods/models.py`

Tabla `periodos_academicos`: id, nombre único, `fecha_inicio`, `fecha_fin` y `estado` (por defecto `ABIERTO`). Dos checks de base: la fecha final no puede ser anterior a la inicial, y el estado solo puede ser `ABIERTO` o `CERRADO`.

### `app/periods/schemas.py`

`PeriodCreate` repite esas reglas en Python: estado literal `ABIERTO` o `CERRADO`, y un validador de modelo que rechaza `fecha_fin` anterior a `fecha_inicio` (eso FastAPI lo convierte en 422). `PeriodUpdate` es igual. `PeriodResponse` suma `id_periodo`.

### `app/periods/services.py`

`get_period` (404 si no existe), `list_periods`, `create_period` y `update_period`. Al crear o actualizar, un nombre repetido o un check de la base se traduce en 409: "El período ya existe o viola una restricción".

### `app/periods/routes.py`

Igual que materias: leer exige login; crear y actualizar exigen administrador. No hay borrado.

---

## Inscripciones

### `app/enrollments/models.py`

Tabla `inscripciones`: id, estudiante, materia y período (claves foráneas), fecha puesta por la base y estado `ACTIVA` o `CANCELADA`. La restricción única `id_usuario + id_materia + id_periodo` impide inscribir dos veces la misma combinación.

### `app/enrollments/schemas.py`

`EnrollmentCreate` pide materia y período (enteros mayores que 0). `id_usuario` es opcional: el administrador puede inscribir a otro; el estudiante no tiene que enviarlo. `EnrollmentResponse` devuelve id, las tres claves, la fecha y el estado.

### `app/enrollments/services.py`

`_has_role` mira si el usuario tiene un rol por nombre.

`get_enrollment` carga la inscripción o lanza 404. La devuelve si quien pide es administrador, si es el propio estudiante, o si es el docente asignado a esa materia. Cualquier otro caso es 403.

`list_enrollments` parte de todas, ordenadas por id. El administrador las ve todas. El docente solo las de materias donde `id_docente` es él. El resto solo las suyas. Luego pagina.

`create_enrollment`:

1. Si no es administrador ni estudiante, 403.
2. El id del estudiante es el del cuerpo solo cuando quien actúa es administrador y lo envió. En cualquier otro caso es el id de quien está autenticado.
3. Si no es administrador e intenta inscribir a otro id, 403.
4. El estudiante debe existir, estar activo y tener rol `ESTUDIANTE`.
5. La materia debe existir y estar activa.
6. El período debe existir, estar `ABIERTO` y el día de hoy debe caer entre `fecha_inicio` y `fecha_fin`.
7. Inserta con estado por defecto `ACTIVA`. Si la combinación ya existe, 409.

`cancel_enrollment` reutiliza la visibilidad de `get_enrollment`, pero cancelar solo puede hacerlo el administrador o el propio estudiante. El docente asignado puede verla y no puede cancelarla. Si ya está `CANCELADA`, la devuelve igual. Si no, cambia el estado y guarda. La fila sigue en la base.

### `app/enrollments/routes.py`

`POST`, `GET` (lista y una) y `DELETE` bajo `/enrollments`. Todas exigen usuario autenticado. El `DELETE` cancela; no borra.

---

## Calificaciones

### `app/grades/models.py`

Tabla `calificaciones`: id, inscripción, nota `Numeric(5, 2)`, tipo y fecha de registro. Checks de base: nota entre 0 y 100, tipo solo `PARCIAL`, `FINAL` o `RECUPERACION`. Una inscripción puede tener varias notas.

### `app/grades/schemas.py`

`GradeCreate` pide inscripción, nota (0–100, hasta 5 dígitos y 2 decimales) y tipo. `GradeUpdate` solo nota y tipo. `GradeResponse` suma id y fecha.

### `app/grades/services.py`

`_can_write` es verdadero para un administrador, o para el docente asignado a la materia de esa inscripción que además tenga rol `DOCENTE`.

`_enrollment` carga la inscripción o lanza 404.

`get_grade` carga la nota. Si el actor no es el estudiante de esa inscripción y tampoco puede escribir, 403.

`list_grades` une notas con inscripciones. El administrador ve todas. El docente ve las de sus materias. El resto ve las de sus propias inscripciones.

`create_grade` exige poder escribir y que la inscripción esté `ACTIVA`. Si está cancelada, 409. Luego inserta.

`update_grade` vuelve a comprobar escritura y que la inscripción siga activa, y solo cambia `nota` y `tipo`.

`delete_grade` solo lo permite un administrador. Borra la fila de verdad (a diferencia de materias e inscripciones) y no devuelve cuerpo.

### `app/grades/routes.py`

CRUD en `/grades`. Crear responde 201. Borrar responde 204 sin contenido. La autorización está en el servicio, no en la ruta: cualquier autenticado puede llamar, y el servicio decide.

---

## Reportes

### `app/reports/schemas.py`

`SubjectReport`: id de materia, nombre, cantidad de inscripciones activas y promedio de notas. El promedio puede ser `null` si no hay notas.

### `app/reports/services.py`

`by_period` solo deja pasar a administrador o docente. Si el período no existe, 404.

Arma dos subconsultas, separadas a propósito para que el conteo de inscripciones no se multiplique por el número de notas:

1. Por materia, cuenta inscripciones del período con estado `ACTIVA`.
2. Por materia, promedia `nota` de esas mismas inscripciones activas.

Luego toma todas las materias y les hace `outerjoin` con ambas. `coalesce(..., 0)` pone 0 cuando una materia no tiene inscripciones. El promedio queda en `null` si no hay notas. El administrador ve todas las materias. El docente solo aquellas donde él es `id_docente`. Ordena por id de materia y construye la lista de `SubjectReport`.

### `app/reports/routes.py`

`GET /api/v1/reports/periods/{period_id}/subjects` exige login y delega en `by_period`.

---

## Esquema de la base

`alembic/env.py` conecta con el mismo `engine` de la aplicación, le pasa `Base.metadata` a Alembic y corre las migraciones dentro de una transacción. `compare_type=True` hace que Alembic compare también los tipos de columna. El `import app.core.models` registra las tablas antes de esa comparación.

`alembic/versions/001_initial.py` es la migración inicial (`down_revision = None`). `upgrade` crea, en este orden: `roles`, `usuarios`, `usuario_rol`, `materias`, `periodos_academicos`, `inscripciones` y `calificaciones`, con las mismas claves, checks y unicidad de los modelos. Además crea dos índices que los modelos no declaran: `ix_inscripciones_materia_periodo` y `ix_calificaciones_inscripcion`. `downgrade` los quita en el orden inverso.

La API no crea tablas al arrancar. Eso lo hace `alembic upgrade head`.

---

## Pruebas

`tests/conftest.py` fija, antes de importar la app, una clave de base y un secreto JWT de prueba. El fixture `client` crea un SQLite en memoria compartido (`StaticPool`), genera las tablas con `create_all`, inserta los tres roles y un usuario por rol con clave `ClaveSegura123!`, y sustituye `get_db` para que la API use esa base. Al terminar, quita el reemplazo y cierra el engine.

`tests/test_api.py` recorre el flujo: el estudiante ve su perfil sin `password_hash` y no puede listar usuarios (403); el admin crea materia y período abierto; el estudiante se inscribe y un segundo intento da 409; el docente pone 87.50 y el estudiante la lee, pero no puede calificar (403); una nota 101 da 422; el reporte del docente muestra 1 inscripción activa y promedio 87.50. Una segunda prueba comprueba que un token basura da 401 y que un período con fecha final anterior a la inicial da 422.

---

## Contenedores

`Dockerfile` parte de Python 3.12 slim, evita archivos `.pyc` y deja la salida sin buffer. Instala primero `requirements.lock` (para aprovechar la caché), crea el grupo `appgroup` y el usuario sin privilegios `appuser`, copia `app`, `alembic` y `alembic.ini`, y arranca como `appuser` con `python -m app.run` en el puerto 8000.

`compose.yaml` tiene tres servicios:

- `db`: PostgreSQL 18, con nombre, usuario y clave obligatorios desde el entorno. El volumen `pgdata` se monta en `/var/lib/postgresql`. El healthcheck usa `pg_isready`. No publica el puerto 5432 al equipo.
- `migrate`: construye la misma imagen y corre `alembic upgrade head` una sola vez, después de que la base esté sana. `restart: "no"` evita que se repita.
- `api`: publica `127.0.0.1:8000` y solo arranca cuando la migración terminó bien. Su healthcheck llama a `/health/live`.

### 🐳 Herramientas de Docker (Para empaquetar tu aplicación)

*   **`Dockerfile`**: Contiene las instrucciones paso a paso para construir la "imagen" de tu aplicación (indica qué versión de Python usar, qué archivos copiar y qué comandos ejecutar para arrancar el proyecto).
*   **`compose.yaml`**: (Antes llamado `docker-compose.yml`). Es el director de orquesta. Sirve para levantar varios contenedores a la vez con un solo comando. Por ejemplo, define que se inicie tu aplicación de Python junto con su base de datos (PostgreSQL o MySQL) y los conecta entre sí.
*   **`.dockerignore`**: Le dice a Docker qué archivos o carpetas de tu computadora no debe subir al contenedor (como archivos temporales o carpetas de configuración locales) para que el contenedor sea ligero y rápido.

---

### ⚙️ Configuración y Bases de Datos

*   **`.env`**: Es un archivo de texto donde se guardan las variables de entorno y credenciales ocultas del proyecto (como contraseñas de bases de datos, claves secretas o puertos de conexión). De esta forma, no expones datos sensibles directamente en el código fuente.
*   **`alembic.ini`**: Es el archivo de configuración de Alembic, una herramienta de Python muy usada para gestionar las "migraciones" de la base de datos. Sirve para llevar un historial y actualizar la estructura de tus tablas de forma organizada a medida que el proyecto cambia.

---

### 🐍 Dependencias de Python (Librerías necesarias)

*   **`requirements.txt`**: Es la lista de todas las librerías externas de Python que tu proyecto necesita para poder ejecutarse (por ejemplo: Flask, FastAPI o Django).
*   **`requirements-dev.txt`**: Contiene librerías adicionales que solo se usan mientras estás programando o haciendo pruebas (como formateadores de código o herramientas de testing), pero que no son necesarias cuando el proyecto ya está publicado en internet.
*   **`requirements.lock`**: Es un archivo que "candadea" las versiones exactas de todas tus librerías y sus dependencias internas. Garantiza que si otra persona (o Docker) instala el proyecto, se descargue exactamente la misma versión que tú usas, evitando que el código falle por actualizaciones inesperadas.

---

### 📁 Documentación y Control de Versiones

*   **`README.md`**: Es el manual de instrucciones del proyecto escrito en formato Markdown. Ahí se explica de qué trata la aplicación, cómo se instala y qué comandos debes ejecutar para ponerla a funcionar.
*   **`.gitignore`**: Le dice al sistema Git qué archivos ignorar por completo para que nunca se suban a tu repositorio de código (por ejemplo, ignora el archivo `.env` para no subir tus contraseñas a GitHub por error).
*   **`docs/` (Carpeta)**: Contiene archivos adicionales de documentación, guías de usuario o manuales de arquitectura del proyecto.
